# 🎬 4DGaussians-Enhanced v1.1: Hybrid Pipeline (Poses + Geometry)

**REVISION:** Sequential pipeline combining:
- **Pose Accuracy** from Calibration Clip (Section 3.2)
- **Geometric Detail** from Extraview Strategy (Section 3.5)

**Pipeline Flow:** Setup → Calib (3.2) → Extraview Recon (3.5) → Alignment/Merge (3.6) → Training


## 📦 Cell 1: Kurulum (Installation)

Tüm bağımlılıkları kurar: 4DGaussians, SAM2, COLMAP, C++ submodule'ler

In [ ]:
# ============================================================
# CELL 1: INSTALLATION (getcwd HATASI DÜZELTİLDİ)
# ============================================================
import os
import sys
import shutil

PROJECT_DIR = "/content/4DGaussians-Enhanced"

# ==========================================
# 🚨 KRİTİK DÜZELTME: GÜVENLİ BÖLGEYE ÇIK
# ==========================================
# Eğer zaten projenin içindeysek, silmeden önce dışarı çıkmalıyız.
# Yoksa "getcwd: cannot access parent directories" hatası alırız.
os.chdir("/content")
print(f"📍 Güvenli ana dizine geçildi: {os.getcwd()}")

# 1. Temizlik (Temiz bir başlangıç için)
if os.path.exists(PROJECT_DIR):
    print(f"🧹 Eski kurulum temizleniyor: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

# 2. Repo'yu Klonla
print("\n📥 Repo klonlanıyor (Loglar açık)...")
!git clone https://github.com/semhfe/4DGaussians-Enhanced.git

# 3. Proje Klasörüne Gir
os.chdir(PROJECT_DIR)
print(f"📂 Proje dizinine girildi: {os.getcwd()}")

# 4. Doğru Branch'e Geç
print("\n🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...")
!git checkout copilot/fix-4dgaussians-enhanced-errors

# 5. Alt Modülleri İndir (KRİTİK ADIM)
print("\n📦 Alt modüller (Submodules) indiriliyor...")
!git submodule update --init --recursive

# 6. requirements.txt Düzenleme
print("\n🔧 'requirements.txt' düzenleniyor (Torch/MMCV temizliği)...")
!sed -i '/torch/d' requirements.txt
!sed -i '/mmcv/d' requirements.txt
!cat requirements.txt | head -n 5

# 7. Bağımlılıkları Yükleme (LOGLAR AÇIK)
print("\n📦 Python kütüphaneleri kuruluyor (Detaylı çıktı)...")
!pip install matplotlib lpips plyfile pytorch_msssim open3d imageio[ffmpeg] opencv-python
!pip install ultralytics supervision huggingface_hub
# SAM2'yi kaynaktan kur
!pip install "git+https://github.com/facebookresearch/sam2.git"

# 8. Setup Scriptini Çalıştırma
print("\n🔧 Setup scripti çalıştırılıyor (C++ Yamaları, Ninja & COLMAP)...")
if os.path.exists("scripts/colab_setup.py"):
    !python scripts/colab_setup.py
else:
    print("❌ HATA: 'scripts/colab_setup.py' dosyası bulunamadı!")
    print("   Lütfen branch isminin doğru olduğundan emin olun.")
    # Dosya yapısını kontrol et
    if os.path.exists("scripts"):
        print(f"   Mevcut dosyalar: {os.listdir('scripts')}")
    else:
        print("   'scripts' klasörü bile yok! Klonlama hatalı olabilir.")

print("\n" + "="*50)
print("✅ Kurulum tamamlandı! (Lütfen yukarıdaki loglarda 'error' olup olmadığını kontrol edin)")
print("="*50)

📍 Güvenli ana dizine geçildi: /content

📥 Repo klonlanıyor (Loglar açık)...
Cloning into '4DGaussians-Enhanced'...
remote: Enumerating objects: 2706, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 2706 (delta 39), reused 50 (delta 21), pack-reused 2625 (from 1)
Receiving objects: 100% (2706/2706), 66.49 MiB | 48.08 MiB/s, done.
Resolving deltas: 100% (1255/1255), done.
📂 Proje dizinine girildi: /content/4DGaussians-Enhanced

🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...
Branch 'copilot/fix-4dgaussians-enhanced-errors' set up to track remote branch 'copilot/fix-4dgaussians-enhanced-errors' from 'origin'.
Switched to a new branch 'copilot/fix-4dgaussians-enhanced-errors'

📦 Alt modüller (Submodules) indiriliyor...
Submodule 'submodules/depth-diff-gaussian-rasterization' (https://github.com/ingra14m/depth-diff-gaussian-rasterization) registered for path 'submodules/depth-diff-gaussian-rasterization'


In [ ]:
# ============================================================
# CELL 1.5: FINAL COMPILATION (C++ Modüllerini Derle)
# ============================================================
import os
import sys

# Proje dizininde olduğumuzdan emin olalım
os.chdir("/content/4DGaussians-Enhanced")

print("🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...")

# 1. Rasterizer Derleme
print("\n📦 Compiling Diff-Gaussian-Rasterization...")
!pip install -e submodules/depth-diff-gaussian-rasterization

# 2. KNN Derleme
print("\n📦 Compiling Simple-KNN...")
!pip install -e submodules/simple-knn

print("\n✅ Derleme tamamlandı! Artık Cell 2'ye geçebilirsiniz.")

🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...

📦 Compiling Diff-Gaussian-Rasterization...
Obtaining file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of diff-gaussian-rasterization==0.0.0 from file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for diff-gaussian-rasterization

📦 Compiling Simple-KNN...
Obtaining file:///content/4DGaussians-Enhanced/submodules/simple-knn
  Preparing m

## 📁 Cell 2: Veri Hazırlama (Data Setup)

Google Drive'ı mount eder, veriyi unzip eder ve formatı doğrular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

print("="*60)
print("📁 Veri Hazırlama")
print("="*60)

# Step 1: Mount Google Drive
print("\n📂 Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("✅ Drive mount edildi")

# Step 2: Configure paths
# BURADAN DÜZENLEYIN: Veri yollarınızı belirtin
DATA_SOURCE = "/content/drive/MyDrive/4DGS_project/input/added_environment.zip"  # Zip dosyası veya klasör yolu
OUTPUT_BASE = "/content/drive/MyDrive/4DGS_project/output"  # Çıktıların kaydedileceği Drive klasörü

# Local processing paths (faster than Drive)
LOCAL_DATA = "/content/data/my_scene"  # Lokal veri klasörü (işleme için)
LOCAL_OUTPUT = "/content/output"  # Lokal çıktı (eğitim için)

# Step 3: Extract or copy data to local disk
os.makedirs(LOCAL_DATA, exist_ok=True)

if DATA_SOURCE.endswith('.zip'):
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Zip dosyası bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📦 Zip açılıyor: {DATA_SOURCE}")
        with zipfile.ZipFile(DATA_SOURCE, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATA)
        print(f"✅ Zip açıldı: {LOCAL_DATA}")
else:
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Klasör bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📂 Veri kopyalanıyor: {DATA_SOURCE} -> {LOCAL_DATA}")
        if os.path.exists(LOCAL_DATA):
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(DATA_SOURCE, LOCAL_DATA)
        print(f"✅ Veri kopyalandı")

# Step 4: Detect data format
print("\n🔍 Veri formatı algılanıyor...")
contents = os.listdir(LOCAL_DATA)
print(f"   İçerik: {contents}")

data_format = None
if 'transforms_train.json' in contents:
    data_format = 'blender'
    print("✅ Format: Blender/NeRF Synthetic")
elif 'sparse' in contents or 'images' in contents:
    data_format = 'colmap'
    print("✅ Format: COLMAP")
elif any('cam' in item for item in contents):
    data_format = 'multicam'
    print("✅ Format: Multi-camera (cam01, cam02, ...)")
elif len([f for f in contents if f.endswith(('.jpg', '.png'))]) > 0:
    data_format = 'raw_images'
    print("✅ Format: Ham resimler (COLMAP gerekli)")
else:
    print("⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.")

# Step 5: Create output directory
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("\n" + "="*60)
print("✅ Veri hazırlama tamamlandı!")
print("="*60)
print(f"\n📁 Lokal veri: {LOCAL_DATA}")
print(f"📁 Lokal çıktı: {LOCAL_OUTPUT}")
print(f"📁 Drive çıktı: {OUTPUT_BASE}")
print(f"\n📊 Format: {data_format}")

if data_format == 'raw_images':
    print("\n⚠️  Ham resimler tespit edildi!")
    print("   Cell 3'ü çalıştırarak COLMAP ile kamera pozlarını hesaplayın")
else:
    print("\n📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")

📁 Veri Hazırlama

📂 Google Drive mount ediliyor...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mount edildi

📦 Zip açılıyor: /content/drive/MyDrive/4DGS_project/input/added_environment.zip
✅ Zip açıldı: /content/data/my_scene

🔍 Veri formatı algılanıyor...
   İçerik: ['added_environment', '__MACOSX']
⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.

✅ Veri hazırlama tamamlandı!

📁 Lokal veri: /content/data/my_scene
📁 Lokal çıktı: /content/output
📁 Drive çıktı: /content/drive/MyDrive/4DGS_project/output

📊 Format: None

📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun


In [ ]:
# ============================================================
# CELL 2.5: COLMAP INSTALLATION (ÖN HAZIRLIK)
# ============================================================
# Bu hücreyi Cell 3'ten ÖNCE çalıştırın.
# Sistemde COLMAP yüklü değilse otomatik olarak kurar.
# ============================================================

import shutil
import os

print("🔍 COLMAP kurulumu kontrol ediliyor...")

# COLMAP komutu sistemde var mı bak
if not shutil.which("colmap"):
    print("📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...")
    try:
        # 1. Paket listesini güncelle (Sessiz mod)
        !apt-get update
        # 2. COLMAP'i kur (Sessiz mod, onay istemeden)
        !apt-get install -y colmap
        print("✅ COLMAP başarıyla kuruldu!")
    except Exception as e:
        print(f"❌ Kurulum sırasında hata oluştu: {e}")
        print("👉 İpucu: '!apt-get install -y colmap' komutunu manuel deneyebilirsiniz.")
else:
    print("✅ COLMAP zaten sistemde yüklü, kuruluma gerek yok.")

# Kurulumu doğrula
print("-" * 30)
print("Sürüm Kontrolü:")
!colmap help | head -n 1

🔍 COLMAP kurulumu kontrol ediliyor...
📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illi

## 📷 Cell 3.2: Phase 1 - Rig Calibration (Perfect Poses)

**Goal:** Extract camera poses from the calibration clip (rig).

**Output:** `/content/output/sparse_calibrated`


In [ ]:
# ============================================================
# CELL 3.2: PHASE 1 - RIG CALIBRATION (PERFECT POSES)
# ============================================================
# Goal: Extract camera poses from a calibration rig clip
# Output: /content/output/sparse_calibrated
# ============================================================

import os
import shutil
import subprocess
import re

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Path to calibration images (8 camera folders: cam01, cam02, ... cam08)
# OR 8 video files that will be extracted
CALIBRATION_INPUT = "/content/drive/MyDrive/4DGS_project/input/calibration_clip"

# Output paths
CALIB_WORKSPACE = "/content/colmap_rig_workspace"
SPARSE_CALIBRATED = "/content/output/sparse_calibrated"

# ==============================================================================
# HYBRID INPUT DETECTION (Images OR Videos)
# ==============================================================================
print("=" * 60)
print("🎯 PHASE 1: RIG CALIBRATION - PERFECT POSES")
print("=" * 60)

def detect_input_type(input_path):
    """Detect if input contains image folders or video files."""
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ Input path not found: {input_path}")
    
    contents = os.listdir(input_path)
    
    # Priority 1: Check for camera subfolders (cam01, cam02, etc.)
    cam_folders = sorted([f for f in contents if os.path.isdir(os.path.join(input_path, f)) 
                          and f.lower().startswith('cam')])
    if len(cam_folders) >= 8:
        print(f"✅ Detected {len(cam_folders)} camera folders (image mode)")
        return "images", cam_folders[:8]
    
    # Priority 2: Check for video files
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    video_files = sorted([f for f in contents if f.lower().endswith(video_extensions)])
    if len(video_files) >= 8:
        print(f"✅ Detected {len(video_files)} video files (video mode)")
        return "videos", video_files[:8]
    
    raise ValueError(f"❌ Could not detect 8 cameras. Found: {contents}")

input_type, inputs = detect_input_type(CALIBRATION_INPUT)

# ==============================================================================
# PREPARE WORKSPACE
# ==============================================================================
print("\n📁 Preparing workspace...")

# Clean workspace
if os.path.exists(CALIB_WORKSPACE):
    shutil.rmtree(CALIB_WORKSPACE)
os.makedirs(CALIB_WORKSPACE, exist_ok=True)

RIG_IMAGES_DIR = os.path.join(CALIB_WORKSPACE, "images", "rig1")
os.makedirs(RIG_IMAGES_DIR, exist_ok=True)

# Map inputs to cam01...cam08
if input_type == "images":
    # Copy/symlink image folders
    for i, folder in enumerate(inputs):
        src = os.path.join(CALIBRATION_INPUT, folder)
        dst = os.path.join(RIG_IMAGES_DIR, f"cam{i+1:02d}")
        shutil.copytree(src, dst)
        print(f"   📂 {folder} → cam{i+1:02d}")
else:
    # Extract frames from videos using ffmpeg
    print("\n🎬 Extracting frames from videos...")
    for i, video in enumerate(inputs):
        src = os.path.join(CALIBRATION_INPUT, video)
        dst = os.path.join(RIG_IMAGES_DIR, f"cam{i+1:02d}")
        os.makedirs(dst, exist_ok=True)
        
        # Extract frames (1 frame per second, adjust as needed)
        cmd = f'ffmpeg -i "{src}" -vf "fps=1" "{dst}/frame_%04d.jpg"'
        subprocess.run(cmd, shell=True, check=True)
        frame_count = len(os.listdir(dst))
        print(f"   🎬 {video} → cam{i+1:02d} ({frame_count} frames)")

# ==============================================================================
# COLMAP RIG CALIBRATION PIPELINE
# ==============================================================================
COLMAP_DB = os.path.join(CALIB_WORKSPACE, "database.db")
COLMAP_SPARSE = os.path.join(CALIB_WORKSPACE, "sparse")
os.makedirs(COLMAP_SPARSE, exist_ok=True)

print("\n" + "=" * 40)
print("📸 COLMAP RIG CALIBRATION")
print("=" * 40)

# 1. Feature Extraction (one camera per folder)
print("\n1️⃣ Feature Extraction (single_camera_per_folder=1)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {RIG_IMAGES_DIR} \
    --ImageReader.single_camera_per_folder 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Sequential Matcher (for rig/video sequences)
print("\n2️⃣ Sequential Matcher...")
!colmap sequential_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper (with rig constraints)
print("\n3️⃣ Mapper (ba_refine_sensor_from_rig=0)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {RIG_IMAGES_DIR} \
    --output_path {COLMAP_SPARSE} \
    --Mapper.ba_refine_sensor_from_rig 0

# Find the model path (COLMAP creates subfolder like '0')
sparse_model = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model):
    sparse_model = COLMAP_SPARSE

# ==============================================================================
# 🔴 SANITY CHECK: RIG INTEGRITY
# ==============================================================================
print("\n" + "=" * 40)
print("🔍 SANITY CHECK: RIG INTEGRITY")
print("=" * 40)

# Convert to TXT for parsing
!colmap model_converter \
    --input_path {sparse_model} \
    --output_path {sparse_model} \
    --output_type TXT

# Check camera count
images_txt = os.path.join(sparse_model, "images.txt")
if os.path.exists(images_txt):
    with open(images_txt, 'r') as f:
        lines = [l for l in f.readlines() if not l.startswith('#') and l.strip()]
    # Each image takes 2 lines in COLMAP format
    num_cameras = len(lines) // 2
    print(f"   📷 Registered cameras: {num_cameras}")
    
    if num_cameras != 8:
        raise RuntimeError(f"🔴 CRITICAL ERROR: Expected 8 cameras, got {num_cameras}!")
    print("   ✅ All 8 cameras registered successfully")
else:
    raise FileNotFoundError("🔴 CRITICAL ERROR: images.txt not found!")

# Check reprojection error (parse from COLMAP output or run bundle_adjuster)
# For now, we'll trust the mapper output. Advanced: parse mapper logs.
print("   ✅ Rig calibration passed sanity checks")

# ==============================================================================
# SAVE CALIBRATED POSES
# ==============================================================================
print("\n📦 Saving calibrated poses...")

if os.path.exists(SPARSE_CALIBRATED):
    shutil.rmtree(SPARSE_CALIBRATED)
shutil.copytree(sparse_model, SPARSE_CALIBRATED)

# Also save as BIN
!colmap model_converter \
    --input_path {SPARSE_CALIBRATED} \
    --output_path {SPARSE_CALIBRATED} \
    --output_type BIN

print(f"\n✅ PHASE 1 COMPLETE: Perfect Poses Acquired!")
print(f"📍 Saved to: {SPARSE_CALIBRATED}")
print("=" * 60)


## ☁️ Cell 3.5: Phase 2 - Extraview Reconstruction (Dense Geometry)

**Goal:** Build dense point cloud from subject scene + extraviews.

**Input:** Main subject images + extra views (NOT calibration clip)

**Output:** `/content/output/sparse_geometric_source`


In [ ]:
# ============================================================
# CELL 3.5: PHASE 2 - EXTRAVIEW RECONSTRUCTION (DENSE GEOMETRY)
# ============================================================
# Goal: Build dense point cloud from subject scene + extraviews
# Input: Main subject images + extra views (NOT calibration clip)
# Output: /content/output/sparse_geometric_source
# ============================================================

import os
import shutil

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Path to SUBJECT images (main views + extraviews)
# This should contain: image00.jpg, image01.jpg, ... AND extra00.jpg, extra01.jpg, etc.
SUBJECT_IMAGES_INPUT = "/content/drive/MyDrive/4DGS_project/input/work_allviews/images"

# Output paths
EXTRAVIEW_WORKSPACE = "/content/colmap_extraview_workspace"
SPARSE_GEOMETRIC = "/content/output/sparse_geometric_source"
DENSE_OUTPUT = os.path.join(EXTRAVIEW_WORKSPACE, "dense")

print("=" * 60)
print("🎯 PHASE 2: EXTRAVIEW RECONSTRUCTION - DENSE GEOMETRY")
print("=" * 60)

# ==============================================================================
# DENSE RECONSTRUCTION SETTINGS
# ==============================================================================
PM_WINDOW_RADIUS = 5
PM_NUM_ITERATIONS = 5
PM_GEOM_CONSISTENCY = 1
SF_MIN_NUM_PIXELS = 5
SF_MAX_REPROJ_ERROR = 2
SF_MAX_DEPTH_ERROR = 0.01

# ==============================================================================
# PREPARE WORKSPACE
# ==============================================================================
print("\n📁 Preparing extraview workspace...")

if os.path.exists(EXTRAVIEW_WORKSPACE):
    shutil.rmtree(EXTRAVIEW_WORKSPACE)
os.makedirs(EXTRAVIEW_WORKSPACE, exist_ok=True)
os.makedirs(DENSE_OUTPUT, exist_ok=True)

COLMAP_DB = os.path.join(EXTRAVIEW_WORKSPACE, "database.db")
COLMAP_SPARSE = os.path.join(EXTRAVIEW_WORKSPACE, "sparse")
os.makedirs(COLMAP_SPARSE, exist_ok=True)

# ==============================================================================
# COLMAP RECONSTRUCTION (Fresh - NOT using calibrated poses yet)
# ==============================================================================
print("\n" + "=" * 40)
print("📸 COLMAP EXTRAVIEW RECONSTRUCTION")
print("=" * 40)

# 1. Feature Extraction
print("\n1️⃣ Feature Extraction (GPU)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Exhaustive Matcher (for maximum point coverage)
print("\n2️⃣ Exhaustive Matcher (GPU)...")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper
print("\n3️⃣ Mapper (Sparse Reconstruction)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --output_path {COLMAP_SPARSE}

# Find sparse model path
sparse_model = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model):
    sparse_model = COLMAP_SPARSE

# 4. Image Undistorter (Dense preparation)
print("\n4️⃣ Image Undistorter (Dense Prep)...")
!colmap image_undistorter \
    --image_path {SUBJECT_IMAGES_INPUT} \
    --input_path {sparse_model} \
    --output_path {DENSE_OUTPUT} \
    --output_type COLMAP \
    --max_image_size 2000

# 5. Patch Match Stereo (Depth Maps)
print(f"\n5️⃣ Patch Match Stereo (GPU)...")
print(f"   ⚙️ Window={PM_WINDOW_RADIUS}, Iters={PM_NUM_ITERATIONS}, GeomCheck={PM_GEOM_CONSISTENCY}")
!colmap patch_match_stereo \
    --workspace_path {DENSE_OUTPUT} \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency {PM_GEOM_CONSISTENCY} \
    --PatchMatchStereo.window_radius {PM_WINDOW_RADIUS} \
    --PatchMatchStereo.num_iterations {PM_NUM_ITERATIONS} \
    --PatchMatchStereo.gpu_index 0

# 6. Stereo Fusion (Dense Point Cloud)
print(f"\n6️⃣ Stereo Fusion (Dense Cloud)...")
print(f"   ⚙️ MinPixels={SF_MIN_NUM_PIXELS}, MaxReproj={SF_MAX_REPROJ_ERROR}, MaxDepthErr={SF_MAX_DEPTH_ERROR}")
FUSED_PLY = os.path.join(DENSE_OUTPUT, "fused.ply")
!colmap stereo_fusion \
    --workspace_path {DENSE_OUTPUT} \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path {FUSED_PLY} \
    --StereoFusion.min_num_pixels {SF_MIN_NUM_PIXELS} \
    --StereoFusion.max_reproj_error {SF_MAX_REPROJ_ERROR} \
    --StereoFusion.max_depth_error {SF_MAX_DEPTH_ERROR}

# ==============================================================================
# SAVE GEOMETRIC SOURCE
# ==============================================================================
print("\n📦 Saving geometric source model...")

# Convert to TXT
!colmap model_converter \
    --input_path {sparse_model} \
    --output_path {sparse_model} \
    --output_type TXT

if os.path.exists(SPARSE_GEOMETRIC):
    shutil.rmtree(SPARSE_GEOMETRIC)
shutil.copytree(sparse_model, SPARSE_GEOMETRIC)

# Also copy dense PLY
if os.path.exists(FUSED_PLY):
    shutil.copy(FUSED_PLY, os.path.join(SPARSE_GEOMETRIC, "fused.ply"))
    ply_size = os.path.getsize(FUSED_PLY) / (1024 * 1024)
    print(f"   ☁️ Dense PLY: {ply_size:.2f} MB")

print(f"\n✅ PHASE 2 COMPLETE: Dense Geometry Acquired!")
print(f"📍 Sparse: {SPARSE_GEOMETRIC}")
print(f"☁️ Dense PLY: {FUSED_PLY}")
print("=" * 60)


## 🔀 Cell 3.6: Phase 3 - Alignment & Fusion (Final Model)

**Goal:** Align geometry to rig poses and create final training dataset.

**Input:**
- `sparse_calibrated` (perfect poses from rig)
- `sparse_geometric_source` (dense points from extraview)

**Output:** Final training-ready model


In [ ]:
# ============================================================
# CELL 3.6: PHASE 3 - ALIGNMENT & FUSION
# ============================================================
# Goal: Align "High Detail Geometry" to "Accurate Rig Poses"
# Input: 
#   - sparse_calibrated (perfect poses from rig)
#   - sparse_geometric_source (dense points from extraview)
# Output: Final training-ready model
# ============================================================

import os
import shutil

# ==============================================================================
# PATHS
# ==============================================================================
SPARSE_CALIBRATED = "/content/output/sparse_calibrated"  # Perfect Poses (from 3.2)
SPARSE_GEOMETRIC = "/content/output/sparse_geometric_source"  # Dense Points (from 3.5)
SPARSE_ALIGNED = "/content/output/sparse_aligned"  # Aligned geometry
FINAL_OUTPUT = "/content/data/my_scene/colmap_output/sparse/0"  # Training target

print("=" * 60)
print("🎯 PHASE 3: ALIGNMENT & FUSION")
print("=" * 60)

# ==============================================================================
# STEP 1: ALIGNMENT (Model Aligner)
# ==============================================================================
print("\n" + "=" * 40)
print("🔄 STEP 1: MODEL ALIGNMENT")
print("=" * 40)
print(f"   📥 Input (Geometry): {SPARSE_GEOMETRIC}")
print(f"   📐 Reference (Poses): {SPARSE_CALIBRATED}")

if os.path.exists(SPARSE_ALIGNED):
    shutil.rmtree(SPARSE_ALIGNED)
os.makedirs(SPARSE_ALIGNED, exist_ok=True)

# Run COLMAP model_aligner
# This aligns the geometric model to the rig coordinate system
!colmap model_aligner \
    --input_path {SPARSE_GEOMETRIC} \
    --ref_path {SPARSE_CALIBRATED} \
    --output_path {SPARSE_ALIGNED} \
    --ref_is_gps 0 \
    --alignment_type ecef

print(f"   ✅ Aligned model saved to: {SPARSE_ALIGNED}")

# ==============================================================================
# STEP 2: FUSION (Merge Poses + Geometry)
# ==============================================================================
print("\n" + "=" * 40)
print("🔀 STEP 2: FUSION (POSES + GEOMETRY)")
print("=" * 40)

# Prepare final output directory
if os.path.exists(FINAL_OUTPUT):
    shutil.rmtree(FINAL_OUTPUT)
os.makedirs(FINAL_OUTPUT, exist_ok=True)

# Step A: Copy POSES from sparse_calibrated (We trust Rig Poses)
print("\n   📋 Step A: Copying POSES from Rig Calibration...")
for pose_file in ["cameras.txt", "cameras.bin", "images.txt", "images.bin"]:
    src = os.path.join(SPARSE_CALIBRATED, pose_file)
    if os.path.exists(src):
        shutil.copy(src, FINAL_OUTPUT)
        print(f"      ✓ {pose_file}")

# Step B: Copy POINTS from sparse_aligned (We trust Extraview Geometry)
print("\n   📋 Step B: Copying POINTS from Aligned Geometry...")
for points_file in ["points3D.txt", "points3D.bin"]:
    src = os.path.join(SPARSE_ALIGNED, points_file)
    if os.path.exists(src):
        shutil.copy(src, FINAL_OUTPUT)
        print(f"      ✓ {points_file}")

# Also copy dense PLY if available
dense_ply_src = os.path.join(SPARSE_GEOMETRIC, "fused.ply")
dense_ply_dst = os.path.join(os.path.dirname(FINAL_OUTPUT), "dense_point_cloud.ply")
if os.path.exists(dense_ply_src):
    shutil.copy(dense_ply_src, dense_ply_dst)
    print(f"\n   ☁️ Dense PLY copied to: {dense_ply_dst}")

# ==============================================================================
# STEP 3: FILTER EXTRAVIEWS FROM FINAL MODEL
# ==============================================================================
print("\n" + "=" * 40)
print("🔍 STEP 3: FILTER EXTRAVIEWS")
print("=" * 40)

# We need to ensure only main cameras (no "extra") are in final images.txt
images_txt_path = os.path.join(FINAL_OUTPUT, "images.txt")

if os.path.exists(images_txt_path):
    # Backup original poses BEFORE filtering for sanity check
    pose_backup = {}
    
    with open(images_txt_path, 'r') as f:
        lines = f.readlines()
    
    # First pass: backup main camera poses
    print("\n   🔒 Backing up main camera poses...")
    data_lines = [l for l in lines if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(data_lines):
        if i + 1 >= len(data_lines):
            break
        metadata_line = data_lines[i]
        points_line = data_lines[i + 1]
        
        parts = metadata_line.strip().split()
        if len(parts) >= 10:
            img_name = parts[-1]
            if "extra" not in img_name.lower():
                # Store pose: QW, QX, QY, QZ, TX, TY, TZ
                pose = [float(parts[j]) for j in range(1, 8)]
                pose_backup[img_name] = pose
                print(f"      ✓ Backed up: {img_name}")
        i += 2
    
    # Second pass: filter out extraviews
    print("\n   🗑️ Filtering extraview images...")
    count_kept = 0
    count_deleted = 0
    
    with open(images_txt_path, 'w') as f_out:
        # Write headers
        for line in lines:
            if line.startswith('#'):
                f_out.write(line)
        
        # Process data lines
        i = 0
        while i < len(data_lines):
            if i + 1 >= len(data_lines):
                break
            metadata_line = data_lines[i]
            points_line = data_lines[i + 1]
            
            parts = metadata_line.strip().split()
            if len(parts) >= 10:
                img_name = parts[-1]
                if "extra" not in img_name.lower():
                    f_out.write(metadata_line)
                    f_out.write(points_line)
                    count_kept += 1
                else:
                    count_deleted += 1
            i += 2
    
    print(f"      ✅ Kept: {count_kept} main cameras")
    print(f"      🗑️ Removed: {count_deleted} extraviews")
    
    # Third pass: verify poses haven't drifted (sanity check)
    print("\n   🔍 Verifying pose integrity...")
    with open(images_txt_path, 'r') as f:
        lines = f.readlines()
    
    data_lines = [l for l in lines if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(data_lines):
        if i + 1 >= len(data_lines):
            break
        metadata_line = data_lines[i]
        parts = metadata_line.strip().split()
        if len(parts) >= 10:
            img_name = parts[-1]
            if img_name in pose_backup:
                new_pose = [float(parts[j]) for j in range(1, 8)]
                old_pose = pose_backup[img_name]
                
                for j, (old_val, new_val) in enumerate(zip(old_pose, new_pose)):
                    diff = abs(old_val - new_val)
                    if diff > 1e-6:
                        raise RuntimeError(
                            f"🔴 CRITICAL ERROR: Main camera poses have drifted!\n"
                            f"   Image: {img_name}\n"
                            f"   Parameter {j}: {old_val} → {new_val} (diff: {diff})"
                        )
        i += 2
    
    print("      ✅ Pose integrity verified - no drift detected")

# ==============================================================================
# STEP 4: CONVERT TO BIN FORMAT
# ==============================================================================
print("\n   🔄 Converting to BIN format...")
!colmap model_converter \
    --input_path {FINAL_OUTPUT} \
    --output_path {FINAL_OUTPUT} \
    --output_type BIN

# ==============================================================================
# FINAL SANITY CHECKS
# ==============================================================================
print("\n" + "=" * 40)
print("✅ FINAL SANITY CHECKS")
print("=" * 40)

# Check all required files exist
required_files = ["cameras.bin", "images.bin", "points3D.bin"]
missing = [f for f in required_files if not os.path.exists(os.path.join(FINAL_OUTPUT, f))]

if missing:
    raise RuntimeError(f"🔴 CRITICAL ERROR: Missing files: {missing}")

print("   ✅ All required .bin files present")

# Check points3D is not empty
points_size = os.path.getsize(os.path.join(FINAL_OUTPUT, "points3D.bin"))
print(f"   📊 points3D.bin size: {points_size / 1024:.2f} KB")

if points_size < 1000:
    print("   ⚠️ WARNING: points3D.bin seems small - may have few points")

# Check images count
images_bin = os.path.join(FINAL_OUTPUT, "images.bin")
images_size = os.path.getsize(images_bin)
print(f"   📊 images.bin size: {images_size / 1024:.2f} KB")

print("\n" + "=" * 60)
print("🎉 PHASE 3 COMPLETE: ALIGNMENT & FUSION SUCCESSFUL!")
print("=" * 60)
print(f"\n📍 Final Model: {FINAL_OUTPUT}")
print("   Contains:")
print("   ├── cameras.bin  (from Rig Calibration)")
print("   ├── images.bin   (from Rig Calibration, filtered)")
print("   └── points3D.bin (from Extraview, aligned)")
print("\n👉 Ready for Training!")


## ✅ Cell 4: Pipeline Sanity Checks

Verify masks and training paths before starting training.


In [ ]:
# ============================================================
# CELL 4: PIPELINE SANITY CHECKS
# ============================================================

import os
import cv2
import numpy as np

# ==============================================================================
# MASK VERIFICATION
# ==============================================================================
MASK_PATH = "/content/data/my_scene/masks"  # Adjust as needed

print("=" * 60)
print("🔍 PIPELINE SANITY CHECKS")
print("=" * 60)

print("\n📋 Checking Masks...")

if not os.path.exists(MASK_PATH):
    raise FileNotFoundError(f"🔴 ERROR: Mask directory not found: {MASK_PATH}")

mask_files = [f for f in os.listdir(MASK_PATH) if f.endswith(('.png', '.jpg'))]

if len(mask_files) == 0:
    raise RuntimeError(f"🔴 ERROR: No mask files found in {MASK_PATH}")

print(f"   📁 Found {len(mask_files)} mask files")

# Check for empty/all-black masks
empty_masks = []
for mask_file in mask_files:
    mask_path = os.path.join(MASK_PATH, mask_file)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        empty_masks.append(mask_file)
        continue
    
    non_zero = np.count_nonzero(mask)
    if non_zero == 0:
        empty_masks.append(mask_file)

if empty_masks:
    print(f"   ⚠️ WARNING: {len(empty_masks)} empty/black masks detected:")
    for m in empty_masks[:5]:
        print(f"      - {m}")
    if len(empty_masks) > 5:
        print(f"      ... and {len(empty_masks) - 5} more")
else:
    print("   ✅ All masks have non-zero content")

# ==============================================================================
# TRAINING PATH VERIFICATION
# ==============================================================================
print("\n📋 Checking Training Paths...")

TRAINING_DATA = "/content/data/my_scene"
SPARSE_MODEL = os.path.join(TRAINING_DATA, "colmap_output/sparse/0")

# Check sparse model
if not os.path.exists(SPARSE_MODEL):
    raise FileNotFoundError(f"🔴 ERROR: Sparse model not found: {SPARSE_MODEL}")

required = ["cameras.bin", "images.bin", "points3D.bin"]
for f in required:
    if not os.path.exists(os.path.join(SPARSE_MODEL, f)):
        raise FileNotFoundError(f"🔴 ERROR: Missing {f} in sparse model")

print(f"   ✅ Sparse model verified: {SPARSE_MODEL}")

# Check images directory
IMAGES_DIR = os.path.join(TRAINING_DATA, "images")
if os.path.exists(IMAGES_DIR):
    img_count = len([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png'))])
    print(f"   ✅ Images directory: {img_count} images")
else:
    print(f"   ⚠️ WARNING: Images directory not found at expected path")

print("\n" + "=" * 60)
print("✅ ALL SANITY CHECKS PASSED - READY FOR TRAINING!")
print("=" * 60)


---

## 📝 Remaining Pipeline Steps

The following cells contain mask generation, training, and rendering steps.


## 🎭 Cell 4: SAM2 Maske Oluşturma

YOLO + SAM2.1 ile otomatik maske oluşturur.

In [ ]:
%%bash
# --- 1) SAM2 temizle
pip uninstall -y sam2 SAM-2 || true
rm -rf /content/sam2

# --- 2) Clone + install (resmi öneri: pip install -e .)
git clone https://github.com/facebookresearch/sam2.git /content/sam2
cd /content/sam2
pip  install -e ".[notebooks]"   # jupyter/matplotlib bağımlılıkları dahil

# --- 3) Checkpoint indir (SAM2.1)
mkdir -p /content/sam2/checkpoints
wget  -O /content/sam2/checkpoints/sam2.1_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

# --- 4) Hızlı smoke test
python - << 'PY'
from sam2.build_sam import build_sam2_video_predictor
print("SAM2 import OK")
PY

Obtaining file:///content/sam2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for SAM-2 (pyproject.toml): started
  Building editable for SAM-2 (pyproject.toml): finished with status 'done'
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp311-cp311-linux_x86_64.whl size=13851 sha256=5ebda0864e8deaeee34f77ff000ef46323af7b090f2373c10af9ca6547ecc244
  Stored in directory: /tmp/pip-ephem-wheel-cache-bsqfq99b/wheels/76/dc/37/006d341f6080de50c00d031747ee8a1a03f3fb513175bce1c0
Successfully built SAM-2
SAM2 import OK


--2025-12-22 13:51:40--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.33.67.77, 13.33.67.73, 13.33.67.42, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.33.67.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 898083611 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘/content/sam2/checkpoints/sam2.1_hiera_large.pt’

     0K .......... .......... .......... .......... ..........  0% 1.48M 9m40s
    50K .......... .......... .......... .......... ..........  0% 1.90M 8m36s
   100K .......... .......... .......... .......... ..........  0% 4.69M 6m45s
   150K .......... .......... .......... .......... ..........  0% 2.97M 6m16s
   200K .......... .......... .......... .......... ..........  0% 6.53M 5m27s
   250K .......... .......... .......... .......... ..........  0% 6.59M 4m54s
   300K .......... .......... .......... .......... ..

In [ ]:
import os, glob
from PIL import Image
import torch
from sam2.build_sam import build_sam2_video_predictor

ROOT = "/content/data/my_scene"              # <-- senin root
CAMS = [f"cam{i:02d}" for i in range(1,9)]     # cam00..cam07
SRC_BASE = os.path.join(ROOT, "added_environment")   # ör: frames_by_cam/cam00/*.png
DST_BASE = os.path.join(ROOT, "_sam2_frames")    # çıkış jpg frame klasörleri
OUT_BASE = os.path.join(ROOT, "masks_sam2")      # mask çıkışı

CKPT = "/content/sam2/checkpoints/sam2.1_hiera_large.pt"
CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"

predictor = build_sam2_video_predictor(CFG, CKPT, device="cuda")

def prep_frames(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    frames = []
    for ext in ("*.png","*.jpg","*.jpeg","*.PNG","*.JPG","*.JPEG"):
        frames += glob.glob(os.path.join(src_dir, ext))
    frames = sorted(frames)
    assert frames, f"Frame yok: {src_dir}"
    for i,f in enumerate(frames):
        out = os.path.join(dst_dir, f"{i:05d}.jpg")
        Image.open(f).convert("RGB").save(out, quality=95)
    return len(frames)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    for cam in CAMS:
        src_dir = os.path.join(SRC_BASE, cam)
        dst_dir = os.path.join(DST_BASE, cam)
        out_dir = os.path.join(OUT_BASE, cam)
        os.makedirs(out_dir, exist_ok=True)

        n = prep_frames(src_dir, dst_dir)

        state = predictor.init_state(video_path=dst_dir)
        predictor.reset_state(state)

        # TODO: burada prompt vermen şart:
        # - frame_idx=0’da YOLO bbox (x0,y0,x1,y1) ver
        # - predictor.add_new_points_or_box(...) çağır
        # - predictor.propagate_in_video(state) ile tüm framelere yay
        # - çıkan maskeleri out_dir/00000.png diye yaz

        print(cam, "OK, frames:", n)


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.15it/s]


cam01 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]


cam02 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]


...

cam06 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.15it/s]


cam07 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.27it/s]


cam08 OK, frames: 66


In [ ]:
from ultralytics import YOLO
import os
import numpy as np

# 1) YOLO modelin (senin notebooktaki path neyse onu koy)
yolo = YOLO("/content/yolov8m.pt")   # <-- düzelt

# 2) Hangi sınıfı segment edeceğiz?
# Modelinde sınıf adları varsa bunu kullan:
TARGET_CLASS_NAME = "person"  # <-- örnek: "person", "bag", "camera" vs.

DST_BASE = "/content/data/my_scene/_sam2_frames"  # senin jpg frame klasörün

def pick_bbox_from_yolo(img_path: str):
    r = yolo(img_path, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None

    # class name -> class id bul
    names = r.names  # {id: "name"}
    target_ids = [cid for cid, n in names.items() if n == TARGET_CLASS_NAME]
    if not target_ids:
        raise ValueError(f"YOLO modelinde '{TARGET_CLASS_NAME}' diye class yok. names={names}")

    xyxy = r.boxes.xyxy.cpu().numpy()   # (N,4)
    conf = r.boxes.conf.cpu().numpy()   # (N,)
    cls  = r.boxes.cls.cpu().numpy().astype(int)  # (N,)

    # hedef sınıfa filtre
    mask = np.isin(cls, target_ids)
    if not mask.any():
        return None

    xyxy_f = xyxy[mask]
    conf_f = conf[mask]

    # en yüksek confidence bbox’u seç
    i = int(np.argmax(conf_f))
    return xyxy_f[i].tolist()  # [x0,y0,x1,y1]

BBOX_BY_CAM = {}
for cam in CAMS:
    img0 = os.path.join(DST_BASE, cam, "00000.jpg")
    bbox = pick_bbox_from_yolo(img0)
    if bbox is None:
        raise RuntimeError(f"{cam} frame0 YOLO detection yok: {img0}")
    BBOX_BY_CAM[cam] = bbox

print("BBOX_BY_CAM hazır:", BBOX_BY_CAM)


BBOX_BY_CAM hazır: {'cam01': [862.8431396484375, 59.83685302734375, 1144.0843505859375, 1030.540283203125], 'cam02': [790.1964111328125, 53.1405029296875, 1157.690673828125, 1028.44482421875], 'cam03': [733.1131591796875, 60.0640869140625, 1230.744384765625, 974.9313354492188], 'cam04': [724.8649291992188, 45.421142578125, 1177.550048828125, 1020.4345092773438], 'cam05': [781.567626953125, 60.098785400390625, 1066.4482421875, 1030.3677978515625], 'cam06': [727.1566772460938, 62.4686279296875, 1107.7802734375, 1019.4102172851562], 'cam07': [693.6357421875, 63.73516845703125, 1169.1171875, 967.0900268554688], 'cam08': [765.3072509765625, 68.21072387695312, 1216.759033203125, 1012.3613891601562]}


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
import gc

# Yolları senin yapına göre ayarla
OUT_BASE = "/content/data/my_scene/masks_sam2"
DST_BASE = "/content/data/my_scene/_sam2_frames"
# Eğer BBOX_BY_CAM önceki adımda oluşmadıysa hata verecektir, o yüzden kontrol et
if 'BBOX_BY_CAM' not in globals():
    raise RuntimeError("⚠️ Önceki adımı (3. Adım - YOLO) çalıştırıp BBOX_BY_CAM sözlüğünü oluşturmalısın!")

print("="*60)
print("🚀 SAM2 Manuel Maskeleme (Düzeltilmiş 4. Adım)")
print("="*60)

def save_mask_png(mask_logits: np.ndarray, path: str):
    # Logits -> Binary Mask (0 veya 255)
    # Threshold genelde 0.0'dır (Sigmoid öncesi)
    mask_binary = (mask_logits > 0.0).astype(np.uint8) * 255
    # Sıkıştırma yaparak kaydet
    Image.fromarray(mask_binary[0, 0]).save(path)

def run_cam_safe(cam_name):
    print(f"🎥 İşleniyor: {cam_name}...")

    frames_dir = os.path.join(DST_BASE, cam_name)
    out_dir = os.path.join(OUT_BASE, cam_name)
    os.makedirs(out_dir, exist_ok=True)

    # YOLO'dan gelen kutu [x1, y1, x2, y2]
    box = np.array(BBOX_BY_CAM[cam_name], dtype=np.float32)

    # State başlatma
    inference_state = predictor.init_state(video_path=frames_dir)
    predictor.reset_state(inference_state)

    # 1. İlk Kareye Prompt (Kutu) Ver
    # SAM 2.1 API Güncellemesi: 'inference_state' ilk parametre
    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=1,
        box=box
    )

    # 2. Videoda Yayılım (Propagate)
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        # Maskeyi kaydet
        # out_mask_logits shape: (1, H, W) veya (K, H, W)
        mask_path = os.path.join(out_dir, f"{out_frame_idx:05d}.png")

        # Binary'e çevir ve kaydet
        mask_arr = (out_mask_logits[0] > 0.0).cpu().numpy().astype(np.uint8) * 255

        # SAM2 maskeleri bazen (1, H, W) gelir, bazen (H, W).
        if mask_arr.ndim == 3:
            mask_arr = mask_arr[0]

        Image.fromarray(mask_arr).save(mask_path)

    # Hafıza Temizliği
    # predictor.reset_state(inference_state) # State'i temizle
    del inference_state
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ {cam_name} tamamlandı.")

# Tüm kameraları işle
# Eğer CAMS listesi tanımlı değilse, klasörden bul
if 'CAMS' not in globals():
    CAMS = sorted([d for d in os.listdir(DST_BASE) if os.path.isdir(os.path.join(DST_BASE, d))])

for cam in CAMS:
    try:
        run_cam_safe(cam)
    except Exception as e:
        print(f"❌ HATA ({cam}): {e}")
        import traceback
        traceback.print_exc()

print("\n🎉 TÜM İŞLEMLER BİTTİ!")
print(f"📂 Maskeler şurada: {OUT_BASE}")

🚀 SAM2 Manuel Maskeleme (Düzeltilmiş 4. Adım)
🎥 İşleniyor: cam01...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.31it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:  11%|█         | 7/66 [00:01<00:10,  5.41it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam01 tamamlandı.
🎥 İşleniyor: cam02...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:   8%|▊         | 5/66 [00:00<00:10,  5.94it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam02 tamamlandı.
🎥 İşleniyor: cam03...


...

✅ cam06 tamamlandı.
🎥 İşleniyor: cam07...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.02it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:   5%|▍         | 3/66 [00:00<00:08,  7.40it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam07 tamamlandı.
🎥 İşleniyor: cam08...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.06it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:  11%|█         | 7/66 [00:01<00:10,  5.43it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam08 tamamlandı.

🎉 TÜM İŞLEMLER BİTTİ!
📂 Maskeler şurada: /content/data/my_scene/masks_sam2


## 👀 Cell 5: Maske Önizleme

Oluşturulan maskeleri kontrol edin.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob
import os
from ipywidgets import interact, IntSlider

print("="*60)
print("👀 Maske Önizleme (SAM2 Çıktıları)")
print("="*60)

# --- AYARLAR ---
# Orijinal görsellerin olduğu yer
SRC_BASE = "/content/data/my_scene/added_environment"
# Maskelerin olduğu yer (Senin son işlem sonucun)
MASK_BASE = "/content/data/my_scene/masks_sam2"

# Kamera klasörlerini bul
cam_folders = sorted(glob.glob(os.path.join(SRC_BASE, "cam*")))

if not cam_folders:
    print(f"❌ Kamera klasörü bulunamadı: {SRC_BASE}")
else:
    def preview_mask(camera_idx=0, frame_idx=0):
        cam_folder = cam_folders[camera_idx]
        cam_name = os.path.basename(cam_folder)

        # Orijinal Frame dosyalarını bul (PNG veya JPG)
        frame_files = sorted(glob.glob(os.path.join(cam_folder, "frame_*.jpg")))
        if not frame_files:
             frame_files = sorted(glob.glob(os.path.join(cam_folder, "*.jpg")))

        # Eğer JPG yoksa PNG dene
        if not frame_files:
            frame_files = sorted(glob.glob(os.path.join(cam_folder, "*.png")))

        if not frame_files:
            print(f"⚠️ {cam_name} içinde görsel bulunamadı.")
            return

        if frame_idx >= len(frame_files):
            print(f"Frame {frame_idx} sınır dışı (toplam {len(frame_files)})")
            return

        frame_path = frame_files[frame_idx]

        # --- MASKE YOLUNU BULMA (Kritik Düzeltme) ---
        # Senin maskelerin "masks_sam2/camXX/00000.png" formatında
        mask_folder = os.path.join(MASK_BASE, cam_name)
        # Frame indexine göre maske ismi (00000.png, 00001.png...)
        mask_name = f"{frame_idx:05d}.png"
        mask_path = os.path.join(mask_folder, mask_name)

        # Resmi yükle
        image = np.array(Image.open(frame_path).convert('RGB'))

        # Maskeyi yükle
        if os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert('L'))

            # Maske bazen görselden farklı boyutta olabilir (resize olduysa), eşitleyelim
            if mask.shape[:2] != image.shape[:2]:
                 mask = np.array(Image.open(mask_path).resize((image.shape[1], image.shape[0]), Image.NEAREST).convert('L'))

            # Overlay oluştur
            overlay = image.copy()
            # Foreground'u yeşile boya (Maske > 0 ise)
            # SAM2 çıktısı bazen binary (0-255) bazen logits olabilir, 128 güvenli eşiktir
            overlay[:,:,1] = np.where(mask > 128, np.minimum(overlay[:,:,1] + 100, 255), overlay[:,:,1])
            mask_status = "✅ Maske Mevcut"
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            overlay = image
            mask_status = f"❌ Maske Yok: {mask_name}"

        # Görselleştir
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(image)
        axes[0].set_title(f"{cam_name} - Frame {frame_idx}\nOrijinal")
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title(f"Maske\n({mask_status})")
        axes[1].axis('off')

        axes[2].imshow(overlay)
        axes[2].set_title("Overlay\n(Yeşil = İnsan)")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

    # Toplam frame sayısını ilk kameradan alalım
    first_cam_path = cam_folders[0]
    num_frames = len(glob.glob(os.path.join(first_cam_path, "*.jpg")) + glob.glob(os.path.join(first_cam_path, "*.png")))

    # Interaktif önizleme
    print(f"\n📸 {len(cam_folders)} kamera tespit edildi.")
    print(f"🖼️ Yaklaşık {num_frames} kare (frame) var.\n")

    interact(
        preview_mask,
        camera_idx=IntSlider(min=0, max=len(cam_folders)-1, step=1, value=0, description='Kamera No'),
        frame_idx=IntSlider(min=0, max=num_frames-1, step=1, value=0, description='Frame No')
    )

print("\n📝 Eğer yeşil alan insanı doğru kaplıyorsa Cell 6'ya (Eğitim) geçebilirsin.")

👀 Maske Önizleme (SAM2 Çıktıları)

📸 8 kamera tespit edildi.
🖼️ Yaklaşık 66 kare (frame) var.



interactive(children=(IntSlider(value=0, description='Kamera No', max=7), IntSlider(value=0, description='Fram…


📝 Eğer yeşil alan insanı doğru kaplıyorsa Cell 6'ya (Eğitim) geçebilirsin.


## ⚙️ Cell 6: Eğitim Konfigürasyonu

Eğitim parametrelerini ayarlayın.

In [ ]:
import os
import shutil

print("="*60)
print("⚙️  Eğitim Konfigürasyonu ve Veri Hazırlığı")
print("="*60)

# ==============================================================================
# 1. VERİ SETİ HAZIRLIĞI (Maskeleri ve Görselleri Birleştirme)
# ==============================================================================
# COLMAP çıktısının olduğu yer (Ana dataset kökü burası olacak)
DATASET_PATH = "/content/data/my_scene/colmap_output"
# Maskelerin olduğu yer
MASK_SOURCE = "/content/data/my_scene/masks_sam2"
# Orijinal görsellerin olduğu yer
IMAGE_SOURCE = "/content/data/my_scene/added_environment"

print(f"📂 Veri Seti Yolu: {DATASET_PATH}")

# A) MASKELERİ BAĞLA: Eğitim kodunun maskeleri otomatik tanıması için
# 'masks' klasörü dataset içinde olmalı. Kopyalamak yerine Symlink yapıyoruz (Yer kaplamaz).
target_mask_path = os.path.join(DATASET_PATH, "masks")
if os.path.exists(target_mask_path):
    if os.path.islink(target_mask_path):
        os.unlink(target_mask_path) # Eski linki kaldır
    elif os.path.isdir(target_mask_path):
        shutil.rmtree(target_mask_path) # Eski klasörü sil

try:
    os.symlink(MASK_SOURCE, target_mask_path)
    print(f"   ✅ Maskeler bağlandı: {MASK_SOURCE} -> {target_mask_path}")
except OSError as e:
    # Symlink hatası olursa kopyalamayı dene
    print(f"   ⚠️ Symlink yapılamadı, kopyalanıyor... ({e})")
    shutil.copytree(MASK_SOURCE, target_mask_path)

# B) GÖRSELLERİ KONTROL ET
# Colmap output içinde 'images' klasörü olmayabilir, onu da bağlayalım.
target_image_path = os.path.join(DATASET_PATH, "images")
if not os.path.exists(target_image_path):
    try:
        os.symlink(IMAGE_SOURCE, target_image_path)
        print(f"   ✅ Görseller bağlandı: {IMAGE_SOURCE} -> {target_image_path}")
    except OSError:
        shutil.copytree(IMAGE_SOURCE, target_image_path)

# ==============================================================================
# 2. EĞİTİM PRESETLERİ (A100 İçin Güçlendirildi)
# ==============================================================================
PRESETS = {
    "standard": {
        "iterations": 30000,
        "densify_until_iter": 15000,
        "coarse_iterations": 3000,
        "w_fg": 1.0, "w_bg": 0.1,
        "net_width": 64,
        "desc": "Standart (30k iterasyon)"
    },
    "high_quality": {
        "iterations": 50000,
        "densify_until_iter": 25000,
        "coarse_iterations": 5000,
        "w_fg": 2.0, "w_bg": 0.05, # İnsana daha çok odaklan
        "net_width": 128, # Daha geniş ağ (daha iyi deformasyon)
        "desc": "Yüksek Kalite (50k iter, İnsan odaklı)"
    },
    "ultra_quality": {
        "iterations": 80000, # A100 Gücü!
        "densify_until_iter": 40000,
        "coarse_iterations": 8000,
        "w_fg": 2.0, "w_bg": 0.1, # Arka planı neredeyse yok say, insana aban
        "net_width": 128,
        "desc": "ULTRA Kalite (80k iter, A100 Özel)"
    }
}

# --- AYARLAR ---
# Tavsiyem: A100 olduğu için 'high_quality' veya 'ultra_quality' seçmen.
SELECTED_PRESET = "ultra_quality" # @param ["standard", "high_quality", "ultra_quality"]

cfg = PRESETS[SELECTED_PRESET]

# Parametre Değişkenleri (Cell 7 bunları kullanacak)
ITERATIONS = cfg["iterations"]
DENSIFY_UNTIL = cfg["densify_until_iter"]
COARSE_ITERS = cfg["coarse_iterations"]
NET_WIDTH = cfg["net_width"]

# Maske Ağırlıkları (Mask Weighted Loss)
USE_MASK = True
W_FG = cfg["w_fg"]  # Ön plan ağırlığı
W_BG = cfg["w_bg"]  # Arka plan ağırlığı

print("\n" + "="*60)
print(f"🎯 Seçilen Mod: {SELECTED_PRESET.upper()}")
print(f"   📝 {cfg['desc']}")
print("="*60)
print(f"   🔄 Toplam İterasyon: {ITERATIONS}")
print(f"   🧠 Ağ Genişliği: {NET_WIDTH}")
print(f"   🎭 Maske Kullanımı: {USE_MASK}")
print(f"      - Ön Plan Ağırlığı: {W_FG} (İnsan)")
print(f"      - Arka Plan Ağırlığı: {W_BG} (Çevre)")
print(f"   📂 Dataset Yolu: {DATASET_PATH}")
print("\n📝 Sonraki adım: Cell 7 ile eğitimi başlatın (Loglar açık olacak)")

⚙️  Eğitim Konfigürasyonu ve Veri Hazırlığı
📂 Veri Seti Yolu: /content/data/my_scene/colmap_output
   ✅ Maskeler bağlandı: /content/data/my_scene/masks_sam2 -> /content/data/my_scene/colmap_output/masks

🎯 Seçilen Mod: ULTRA_QUALITY
   📝 ULTRA Kalite (80k iter, A100 Özel)
   🔄 Toplam İterasyon: 80000
   🧠 Ağ Genişliği: 128
   🎭 Maske Kullanımı: True
      - Ön Plan Ağırlığı: 2.0 (İnsan)
      - Arka Plan Ağırlığı: 0.1 (Çevre)
   📂 Dataset Yolu: /content/data/my_scene/colmap_output

📝 Sonraki adım: Cell 7 ile eğitimi başlatın (Loglar açık olacak)


## 🚀 Cell 7: Eğitim

Model eğitimini başlatır. Eğitim lokal diskte yapılır, sonunda Drive'a kopyalanır.

In [ ]:
import os
import struct
import shutil
import glob

print("="*60)
print("🎯 HEDEF ODAKLI PATH DÜZELTİCİ (DOĞRU ADRES)")
print("="*60)

# 1. EĞİTİMDE KULLANILAN GERÇEK PATH
# Cell 7'de kullanılan path burası:
REAL_DATASET_PATH = "/content/data/my_scene/colmap_output"
IMAGES_DIR = os.path.join(REAL_DATASET_PATH, "images")
SPARSE_DIR = os.path.join(REAL_DATASET_PATH, "sparse/0")
IMAGES_BIN = os.path.join(SPARSE_DIR, "images.bin")

print(f"📂 Veri Seti: {REAL_DATASET_PATH}")
print(f"📄 Hedef Dosya: {IMAGES_BIN}")

if not os.path.exists(IMAGES_BIN):
    print("❌ HATA: images.bin bulunamadı! Yol yanlış olabilir.")
    # Belki sparse klasörü direkt köktedir, kontrol et
    alt_path = os.path.join(REAL_DATASET_PATH, "sparse/images.bin")
    if os.path.exists(alt_path):
        IMAGES_BIN = alt_path
        print(f"   ⚠️ Dosya şurada bulundu ve güncellendi: {IMAGES_BIN}")
    else:
        raise FileNotFoundError("images.bin hiçbir yerde bulunamadı.")

# 2. GERÇEK KLASÖR YAPISINI TESPİT ET
# Diskte 'cam01', 'cam02' mi var yoksa 'image01' mi?
folder_candidates = sorted([f for f in os.listdir(IMAGES_DIR) if os.path.isdir(os.path.join(IMAGES_DIR, f))])
print(f"📂 Diskteki Klasörler: {folder_candidates}")

# İlk dosya ismini öğren (frame_00001.png mi 00000.png mi?)
sample_folder = os.path.join(IMAGES_DIR, folder_candidates[0])
sample_files = sorted([f for f in os.listdir(sample_folder) if f.endswith(('.png', '.jpg'))])
if not sample_files:
    raise ValueError("❌ Klasörler boş!")
target_filename = sample_files[0] # örn: frame_00001.png
print(f"🖼️ Örnek Dosya Adı: {target_filename}")

# 3. YARDIMCI FONKSİYONLAR
def read_next_bytes(fid, num_bytes, format_char_sequence, endian_character="<"):
    data = fid.read(num_bytes)
    return struct.unpack(endian_character + format_char_sequence, data)

def write_next_bytes(fid, *args):
    fmt = "<" + args[-1]
    data = args[:-1]
    if len(data) == 1 and isinstance(data[0], (list, tuple)):
        data = data[0]
    fid.write(struct.pack(fmt, *data))

# 4. DOSYAYI OKU VE HAFIZAYA AL
print("\n🔄 Dosya okunuyor...")
images_data = []
with open(IMAGES_BIN, "rb") as fid:
    num_reg_images = read_next_bytes(fid, 8, "Q")[0]
    print(f"   Kayıtlı Kamera Sayısı: {num_reg_images}")

    for _ in range(num_reg_images):
        props = read_next_bytes(fid, 64, "I4d3dI")
        image_id = props[0]
        name = ""
        while True:
            char = fid.read(1)
            if char == b"\x00": break
            name += char.decode("utf-8")

        num_p2d = read_next_bytes(fid, 8, "Q")[0]
        points_data = fid.read(num_p2d * 24) # Veriyi atla ama sakla

        images_data.append({
            "props": props,
            "name": name,
            "num_p2d": num_p2d,
            "points": points_data,
            "id": image_id
        })

# ID'ye göre sırala (Eşleştirme için kritik)
images_data.sort(key=lambda x: x["id"])

# 5. VERİLERİ DÜZENLE (YAMA)
print("\n🛠️ Düzeltme Uygulanıyor...")
# Eğer diskteki klasör sayısı ile kayıttaki sayı tutuyorsa sırayla eşleştir
if len(images_data) == len(folder_candidates):
    for i, img in enumerate(images_data):
        real_folder = folder_candidates[i] # örn: cam01

        # ESKİ: frame_00001.png veya image01.png
        # YENİ: cam01/frame_00001.png

        new_name = f"{real_folder}/{target_filename}"
        print(f"   ID {img['id']}: {img['name']}  --->  {new_name}")
        img["name"] = new_name
else:
    print("⚠️ SAYI UYUŞMAZLIĞI! Otomatik eşleştirme riskli olabilir.")
    print(f"   Kayıt: {len(images_data)} vs Disk: {len(folder_candidates)}")
    print("   Mevcut isimleri 'camXX/...' formatına çevirmeyi deniyorum...")

    # Yedek plan: Mevcut isme bakıp klasör uydurmak
    for i, img in enumerate(images_data):
        # image01.png -> cam01/frame...
        # frame_00001.png -> cam01/frame... (Sıraya göre)
        if i < len(folder_candidates):
            new_name = f"{folder_candidates[i]}/{target_filename}"
            print(f"   ID {img['id']}: {img['name']}  --->  {new_name}")
            img["name"] = new_name

# 6. KAYDET
# Önce yedek al
shutil.copy(IMAGES_BIN, IMAGES_BIN + ".bak")
print(f"\n💾 Orijinal yedeklendi: {IMAGES_BIN}.bak")

with open(IMAGES_BIN, "wb") as fid:
    write_next_bytes(fid, len(images_data), "Q")
    for img in images_data:
        write_next_bytes(fid, *img["props"], "I4d3dI")
        fid.write(img["name"].encode("utf-8") + b"\x00")
        write_next_bytes(fid, img["num_p2d"], "Q")
        fid.write(img["points"])

print(f"✅ Dosya başarıyla güncellendi: {IMAGES_BIN}")
print("\n🚀 ŞİMDİ EĞİTİMİ BAŞLATABİLİRSİN (Cell 7)")

🎯 HEDEF ODAKLI PATH DÜZELTİCİ (DOĞRU ADRES)
📂 Veri Seti: /content/data/my_scene/colmap_output
📄 Hedef Dosya: /content/data/my_scene/colmap_output/sparse/0/images.bin
📂 Diskteki Klasörler: ['cam01', 'cam02', 'cam03', 'cam04', 'cam05', 'cam06', 'cam07', 'cam08']
🖼️ Örnek Dosya Adı: frame_00001.png

🔄 Dosya okunuyor...
   Kayıtlı Kamera Sayısı: 8

🛠️ Düzeltme Uygulanıyor...
   ID 33: cam01/frame_00001.png  --->  cam01/frame_00001.png
   ID 34: cam02/frame_00001.png  --->  cam02/frame_00001.png
   ID 35: cam03/frame_00001.png  --->  cam03/frame_00001.png
   ID 36: cam04/frame_00001.png  --->  cam04/frame_00001.png
   ID 37: cam05/frame_00001.png  --->  cam05/frame_00001.png
   ID 38: cam06/frame_00001.png  --->  cam06/frame_00001.png
   ID 39: cam07/frame_00001.png  --->  cam07/frame_00001.png
   ID 40: cam08/frame_00001.png  --->  cam08/frame_00001.png

💾 Orijinal yedeklendi: /content/data/my_scene/colmap_output/sparse/0/images.bin.bak
✅ Dosya başarıyla güncellendi: /content/data/my_scene

In [ ]:
import os
import re

print("="*60)
print("🧬 KAYNAK KOD AMELİYATI (DATASET READER PATCH)")
print("="*60)

# 1. HEDEF DOSYA
repo_path = "/content/4DGaussians-Enhanced"
target_file = os.path.join(repo_path, "scene", "dataset_readers.py")

if not os.path.exists(target_file):
    print("❌ HATA: Hedef dosya bulunamadı!")
    # Alternatif kontrol
    alt_repo = "/content/4DGaussians"
    if os.path.exists(os.path.join(alt_repo, "scene", "dataset_readers.py")):
        target_file = os.path.join(alt_repo, "scene", "dataset_readers.py")
        print(f"   ⚠️ Alternatif repo bulundu: {target_file}")
    else:
        raise FileNotFoundError("dataset_readers.py hiçbir yerde yok.")

print(f"📄 Hedef: {target_file}")

# 2. DOSYAYI OKU
with open(target_file, "r") as f:
    original_code = f.read()

# 3. SORUNLU SATIRI TESPİT ET VE DÜZELT
# Hedef: os.path.join(images_folder, os.path.basename(...)) yapısını bulup
#        os.path.join(images_folder, ...) haline getirmek.

# Regex ile esnek arama
# Bu regex, parantez içindeki değişken adını (capture group 1) yakalar.
pattern = r'os\.path\.join\s*\(\s*images_folder\s*,\s*os\.path\.basename\s*\(([^)]+)\)\s*\)'

match = re.search(pattern, original_code)

if match:
    print("\n🐛 KISITLAYICI KOD BULUNDU!")
    old_code = match.group(0)
    print(f"   Eski: {old_code}")

    # Değişken adı (örn: image.name veya extr.name)
    var_name = match.group(1)

    # Yeni kod (basename fonksiyonunu çıkarıyoruz)
    new_code = f"os.path.join(images_folder, {var_name})"
    print(f"   Yeni: {new_code}")

    # Değiştir
    patched_code = original_code.replace(old_code, new_code)

    # Kaydet
    with open(target_file, "w") as f:
        f.write(patched_code)

    print("\n✅ YAMA BAŞARIYLA UYGULANDI.")
    print("   Kod artık alt klasörleri (camXX/...) olduğu gibi okuyacak.")

else:
    # Eğer regex bulamazsa, manuel string replace deneyelim (B planı)
    print("⚠️ Regex eşleşmedi, B planı (Manuel Replace) deneniyor...")

    replacements = [
        ("os.path.join(images_folder, os.path.basename(image.name))", "os.path.join(images_folder, image.name)"),
        ("os.path.join(images_folder, os.path.basename(extr.name))", "os.path.join(images_folder, extr.name)")
    ]

    patched = False
    for old, new in replacements:
        if old in original_code:
            original_code = original_code.replace(old, new)
            print(f"   ✏️ Düzeltildi: {old} -> {new}")
            patched = True

    if patched:
        with open(target_file, "w") as f:
            f.write(original_code)
        print("\n✅ YAMA BAŞARIYLA UYGULANDI.")
    else:
        print("\n❌ HATA: Değiştirilecek kod bloğu bulunamadı. Dosya zaten yamalı olabilir mi?")

print("\n🚀 SONUÇ: Şimdi Cell 7'yi çalıştırıp eğitimi başlat!")

🧬 KAYNAK KOD AMELİYATI (DATASET READER PATCH)
📄 Hedef: /content/4DGaussians-Enhanced/scene/dataset_readers.py

🐛 KISITLAYICI KOD BULUNDU!
   Eski: os.path.join(images_folder, os.path.basename(extr.name))
   Yeni: os.path.join(images_folder, extr.name)

✅ YAMA BAŞARIYLA UYGULANDI.
   Kod artık alt klasörleri (camXX/...) olduğu gibi okuyacak.

🚀 SONUÇ: Şimdi Cell 7'yi çalıştırıp eğitimi başlat!


In [ ]:
import os
import sys
import time
import shutil
import subprocess
import glob
from datetime import datetime

print("="*60)
print("🛡️ CELL 7: FINAL EĞİTİM BAŞLATICI (DÜZELTİLMİŞ & GÜÇLENDİRİLMİŞ)")
print("="*60)

# ==============================================================================
# 1. KÜTÜPHANELERİ GARANTİLİ YÜKLE
# ==============================================================================
print(f"🐍 Python Yolu: {sys.executable}")
print("📦 Bağımlılıklar kontrol ediliyor...\n")

critical_packages = ["open3d", "plyfile", "lpips", "tqdm", "opencv-python"]
for pkg in critical_packages:
    try:
        subprocess.check_call([sys.executable, "-c", f"import {pkg.split('-')[0].replace('opencv', 'cv2')}"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except:
        print(f"   ⬇️  Yükleniyor: {pkg}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg], stdout=subprocess.DEVNULL)
            print(f"      ✅ Yüklendi: {pkg}")
        except:
            print(f"      ❌ {pkg} yüklenemedi!")
print("   ✅ Tüm kütüphaneler hazır.")

# ==============================================================================
# 2. AYARLAR VE YOLLAR
# ==============================================================================
if 'DATASET_PATH' not in globals():
    TRAIN_SOURCE = "/content/data/my_scene/colmap_output"
else:
    TRAIN_SOURCE = DATASET_PATH

# Çıktı Klasörü
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
LOCAL_OUTPUT = f"/content/output/4DGS_{timestamp}"
SAVE_ITERS = [7000, 15000, 30000, 50000, ITERATIONS]

# ==============================================================================
# 3. 🧪 MASKE TURNUSOL TESTİ (GELİŞMİŞ EŞLEŞTİRME)
# ==============================================================================
print("\n" + "="*60)
print("🧪 MASKE ENTEGRASYON TESTİ")
print("="*60)

if USE_MASK:
    mask_root = os.path.join(TRAIN_SOURCE, "masks")
    img_root = os.path.join(TRAIN_SOURCE, "images")

    # Görselleri bul (JPG veya PNG)
    sample_images = sorted(glob.glob(os.path.join(img_root, "**", "*.jpg"), recursive=True))[:5]
    if not sample_images:
        sample_images = sorted(glob.glob(os.path.join(img_root, "**", "*.png"), recursive=True))[:5]

    if sample_images:
        print(f"   🔬 Örneklem Kontrolü ({len(sample_images)} dosya):")
        match_count = 0

        for img_path in sample_images:
            rel_path = os.path.relpath(img_path, img_root) # örn: cam01/frame_00001.png
            folder = os.path.dirname(rel_path)              # örn: cam01
            fname = os.path.basename(rel_path)              # örn: frame_00001.png
            fname_no_ext = os.path.splitext(fname)[0]       # örn: frame_00001

            # Olası Maske İsimleri (Senaryolar)
            candidates = [
                fname.replace(".jpg", ".png"),                     # frame_00001.png (Birebir)
                fname_no_ext + ".png",                             # frame_00001.png (PNG ise)
                fname_no_ext.split("_")[-1] + ".png" if "_" in fname_no_ext else "JUNK", # 00001.png (Sayısal)
                f"{int(fname_no_ext.split('_')[-1]):05d}.png" if "_" in fname_no_ext and fname_no_ext.split("_")[-1].isdigit() else "JUNK" # 00001.png (Formatlı)
            ]

            found = False
            for cand in candidates:
                if cand == "JUNK": continue
                mask_full_path = os.path.join(mask_root, folder, cand)
                if os.path.exists(mask_full_path):
                    print(f"      ✅ Eşleşme: {fname} -> {cand}")
                    found = True
                    match_count += 1
                    break

            if not found:
                print(f"      ⚠️  Maske yok: {rel_path} (Denenenler: {candidates})")

        if match_count > 0:
            print("\n   ✅ TEST BAŞARILI: Maskeler algılandı.")
            print(f"   ⚖️  Ağırlıklar: FG={W_FG} | BG={W_BG}")
        else:
            print("\n❌ HATA: Hiçbir maske eşleşmedi!")
            print("   Lütfen Cell 6'da maskelerin 'masks_sam2' klasörüne doğru kopyalandığından emin olun.")
            sys.exit(1)
    else:
        print("❌ Dataset içinde görüntü dosyası bulunamadı!")
        sys.exit(1)
else:
    print("⚠️ USE_MASK = False. Maskesiz eğitim.")

# ==============================================================================
# 4. KOMUT OLUŞTURMA & ÇALIŞTIRMA
# ==============================================================================
work_dir = "/content/4DGaussians-Enhanced"

# Argüman listesi
cmd_args = [
    sys.executable, "train.py",
    "--source_path", TRAIN_SOURCE,
    "--model_path", LOCAL_OUTPUT,
    "--images", "images",
    "--iterations", str(ITERATIONS),
    "--coarse_iterations", str(COARSE_ITERS),
    "--densify_until_iter", str(DENSIFY_UNTIL),
    "--net_width", str(NET_WIDTH),
    "--batch_size", "1",
    "--resolution", "1",
    "--save_iterations", *map(str, SAVE_ITERS)
    # NOT: '--quiet False' parametresi SİLİNDİ (Hataya sebep oluyordu)
    # Parametre vermemek zaten logları açar.
]

if USE_MASK:
    cmd_args.append("--use_mask_loss")
    cmd_args.append("--w_fg"); cmd_args.append(str(W_FG))
    cmd_args.append("--w_bg"); cmd_args.append(str(W_BG))

print("\n" + "="*60)
print(f"🚀 EĞİTİM START ALIYOR (A100 - {ITERATIONS} Iterasyon)")
print("="*60)
print(f"📂 Çıktı: {LOCAL_OUTPUT}")

start_time = time.time()

# Subprocess ile çalıştır
process = subprocess.Popen(
    cmd_args,
    cwd=work_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

# LOG OKUMA (Eğitim başladığında burası akacak)
for line in process.stdout:
    print(line, end="")

process.wait()

# ==============================================================================
# 5. BİTİŞ & YEDEKLEME
# ==============================================================================
duration = time.time() - start_time
hours = duration // 3600
minutes = (duration % 3600) // 60

print("\n" + "="*60)
if process.returncode == 0:
    print(f"✅ EĞİTİM TAMAMLANDI! (Süre: {int(hours)}sa {int(minutes)}dk)")

    DRIVE_ROOT = "/content/drive/MyDrive/4DGS_project/output"
    drive_target = os.path.join(DRIVE_ROOT, f"scene_{timestamp}")
    print(f"📤 Drive Yedekleme: {drive_target}")

    try:
        if not os.path.exists(DRIVE_ROOT): os.makedirs(DRIVE_ROOT)
        shutil.copytree(LOCAL_OUTPUT, drive_target)
        print("✅ Başarıyla yedeklendi.")
    except Exception as e:
        print(f"❌ Yedekleme hatası: {e}")
else:
    print(f"❌ EĞİTİM HATA İLE SONLANDI (Kod: {process.returncode})")

print("="*60)

Streaming output truncated to the last 5000 lines.
Training progress: 100%|██████████| 80000/80000 [2:47:00<00:00,  7.98it/s, Loss=0.0018994, psnr=48.57, point=265166]
data loading done [22/12 15:37:30]

[ITER 3000] Evaluating test: L1 0.0837966650724411 PSNR 15.692878723144531 [22/12 15:42:47]

[ITER 3000] Evaluating train: L1 0.005239250275361187 PSNR 34.76992595897001 [22/12 15:42:54]
reset opacity [22/12 15:42:54]
reset opacity [22/12 15:48:24]

[ITER 7000] Evaluating test: L1 0.09106110781431198 PSNR 15.426862716674805 [22/12 15:50:24]

[ITER 7000] Evaluating train: L1 0.002909019575728213 PSNR 41.499301461612475 [22/12 15:50:27]

[ITER 7000] Saving Gaussians [22/12 15:50:27]
reset opacity [22/12 15:54:20]
reset opacity [22/12 16:00:30]

[ITER 14000] Evaluating test: L1 0.09185090661048889 PSNR 15.379354476928711 [22/12 16:04:52]

[ITER 14000] Evaluating train: L1 0.0024364181684658807 PSNR 44.50333830889534 [22/12 16:04:55]

[ITER 15000] Saving Gaussians [22/12 16:07:02]
reset op

## 🎥 Cell 8: Render

Eğitilmiş modelden video render eder.

In [ ]:
import glob
from IPython.display import Video, display

print("="*60)
print("🎥 Video Render")
print("="*60)

# Render komutu
render_cmd = f"""python /content/4DGaussians-Enhanced/render.py \
    --source_path {LOCAL_DATA} \
    --model_path {LOCAL_OUTPUT} \
    --iteration {ITERATIONS}"""

print(f"\n📝 Komut:")
print(render_cmd)
print()

!{render_cmd}

# Render edilen videoyu bul
video_files = glob.glob(os.path.join(LOCAL_OUTPUT, "**/*.mp4"), recursive=True)

if video_files:
    print("\n" + "="*60)
    print("✅ Render tamamlandı!")
    print("="*60)
    print(f"\n📹 Video: {video_files[0]}")

    # Videoyu göster
    print("\n📺 Video oynatılıyor...\n")
    display(Video(video_files[0], width=800))

    # Drive'a kopyala
    scene_name = os.path.basename(LOCAL_DATA)
    drive_output = os.path.join(OUTPUT_BASE, scene_name)

    print(f"\n📤 Video Drive'a kopyalanıyor: {drive_output}")
    # Model zaten kopyalandı, sadece render klasörünü güncelle
    render_dir_local = os.path.dirname(video_files[0])
    render_dir_drive = os.path.join(drive_output, os.path.basename(render_dir_local))

    if os.path.exists(render_dir_drive):
        shutil.rmtree(render_dir_drive)
    shutil.copytree(render_dir_local, render_dir_drive)
    print(f"✅ Video Drive'a kopyalandı")
else:
    print("\n⚠️  Video dosyası bulunamadı. Çıktı klasörünü kontrol edin.")

print("\n📝 Sonraki adım: Cell 9 ile PLY export yapın (opsiyonel)")

## 💾 Cell 9: PLY Export (Opsiyonel)

Frame başına 3D Gaussian point cloud'ları export eder.

In [ ]:
import os
import sys
import shutil
import glob
import subprocess

print("="*60)
print("💾 CELL 9: PLY EXPORT (NET_WIDTH DÜZELTİLMİŞ)")
print("="*60)

EXPORT_PLY = True  # @param {type:"boolean"}

# --- YOLLARIN TANIMLANMASI ---
# 1. Dataset
if 'DATASET_PATH' not in globals():
    source_path = "/content/data/my_scene/colmap_output"
else:
    source_path = DATASET_PATH

# 2. Model Çıktısı
if 'LOCAL_OUTPUT' not in globals() or not os.path.exists(LOCAL_OUTPUT):
    output_root = "/content/output"
    if os.path.exists(output_root):
        all_outputs = sorted(glob.glob(os.path.join(output_root, "4DGS_*")))
        if all_outputs:
            LOCAL_OUTPUT = all_outputs[-1]
            print(f"⚠️ LOCAL_OUTPUT güncellendi: {LOCAL_OUTPUT}")
        else:
            print("❌ HATA: Çıktı klasörü bulunamadı!")
            EXPORT_PLY = False
    else:
        print("❌ HATA: /content/output bulunamadı!")
        EXPORT_PLY = False

model_path = LOCAL_OUTPUT

# 3. Model Parametreleri (Hata Çözücü)
# Hataya göre checkpoint 128 boyutunda. Bunu script'e bildirmeliyiz.
NET_WIDTH = 128

if EXPORT_PLY:
    print(f"📂 Kaynak: {source_path}")
    print(f"📂 Model: {model_path}")
    print(f"🧠 Ağ Genişliği: {NET_WIDTH} (Checkpoint ile eşleşmeli)")

    # Çalışma dizini
    work_dir = "/content/4DGaussians-Enhanced"

    # --- DÜZELTME BURADA ---
    # '--net_width 128' eklendi.
    cmd_args = [
        sys.executable, "export_perframe_3DGS.py",
        "--source_path", source_path,
        "--model_path", model_path,
        "--iteration", str(ITERATIONS),
        "--skip_video",
        "--net_width", str(NET_WIDTH) # <--- KRİTİK EKLEME
    ]

    print("\n📝 Export işlemi başlıyor (Bu işlem biraz sürebilir)...")
    print("-" * 60)

    try:
        process = subprocess.Popen(
            cmd_args,
            cwd=work_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            universal_newlines=True
        )

        for line in process.stdout:
            print(line, end="")

        process.wait()

        if process.returncode != 0:
            print(f"\n❌ Export hata kodu: {process.returncode}")
        else:
            print("\n✅ Export komutu başarıyla tamamlandı.")

    except Exception as e:
        print(f"\n❌ Hata: {e}")

    # --- KONTROL VE DRIVE YEDEKLEME ---
    ply_source_dir = os.path.join(model_path, "per_frame_ply")

    if os.path.exists(ply_source_dir):
        ply_files = glob.glob(os.path.join(ply_source_dir, "*.ply"))

        print("\n" + "="*60)
        print(f"💾 {len(ply_files)} karelik 4D PLY serisi bulundu!")
        print(f"📁 Konum: {ply_source_dir}")

        # Drive Hedefi
        model_name = os.path.basename(model_path)
        drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")

        print(f"\n📤 Drive'a aktarılıyor: {drive_ply_dir}")

        if not os.path.exists(os.path.dirname(drive_ply_dir)):
            os.makedirs(os.path.dirname(drive_ply_dir), exist_ok=True)

        if os.path.exists(drive_ply_dir):
            shutil.rmtree(drive_ply_dir)

        shutil.copytree(ply_source_dir, drive_ply_dir)
        print("✅ Kopyalama tamamlandı!")
        print("="*60)
        print("💡 İPUCU: Dosyalar Drive'a yüklendiğinde 'SuperSplat' sitesine sürükleyip bırakarak izleyebilirsin.")

    else:
        print(f"\n⚠️ 'per_frame_ply' klasörü oluşmadı.")
        print("   Loglarda 'traceback' hatası var mı kontrol edin.")

else:
    print("⏭️  Export atlandı.")

💾 CELL 9: PLY EXPORT (NET_WIDTH DÜZELTİLMİŞ)
📂 Kaynak: /content/data/my_scene/colmap_output
📂 Model: /content/output/4DGS_20251222_1530
🧠 Ağ Genişliği: 128 (Checkpoint ile eşleşmeli)

📝 Export işlemi başlıyor (Bu işlem biraz sürebilir)...
------------------------------------------------------------
  File "/content/4DGaussians-Enhanced/export_perframe_3DGS.py", line 113
    points, scales_final, rotations_final, opacity_final, shs_final = get_state_at_time(gaussians, viewpoint)
    ^^^^^^
IndentationError: expected an indented block after 'for' statement on line 110

❌ Export hata kodu: 1

⚠️ 'per_frame_ply' klasörü oluşmadı.
   Loglarda 'traceback' hatası var mı kontrol edin.


In [ ]:
import os
import sys
import shutil
import glob
import subprocess

print("="*60)
print("🚑 SCRIPT TAMİRİ VE EXPORT BAŞLATMA")
print("="*60)

# --- 1. SCRIPT AMELİYATI (PATCH) ---
script_path = "/content/4DGaussians-Enhanced/export_perframe_3DGS.py"
print(f"🔧 Hedef Dosya: {script_path}")

with open(script_path, "r") as f:
    code = f.read()

# Eski Hatalı Satır
old_line = "for index, viewpoint in enumerate(scene.getTestCameras()):"

# Yeni Mantıklı Blok
# Önce Test'e bak, boşsa Train'i al.
new_block = """
    # YAMA: Kameraları akıllı seç
    cameras = scene.getTestCameras()
    if len(cameras) == 0:
        print("⚠️ Test kamerası bulunamadı, Eğitim kameraları (Train) kullanılıyor...")
        cameras = scene.getTrainCameras()

    # Sıralı olması için isme veya zamana göre sıralayalım (Opsiyonel ama iyi olur)
    # cameras.sort(key=lambda x: x.uid)

    for index, viewpoint in enumerate(cameras):
"""

if old_line in code:
    print("   🐛 Hatalı döngü bulundu, düzeltiliyor...")
    new_code = code.replace(old_line, new_block)

    with open(script_path, "w") as f:
        f.write(new_code)
    print("   ✅ Yama başarıyla uygulandı!")
else:
    print("   ℹ️ Yama zaten uygulanmış veya kod farklı.")

# --- 2. EXPORT İŞLEMİNİ BAŞLAT ---
print("\n" + "="*60)
print("🎬 EXPORT START")
print("="*60)

# Yolları Tanımla
if 'DATASET_PATH' not in globals():
    source_path = "/content/data/my_scene/colmap_output"
else:
    source_path = DATASET_PATH

if 'LOCAL_OUTPUT' not in globals() or not os.path.exists(LOCAL_OUTPUT):
    # Output klasörünü bulmaya çalış
    output_root = "/content/output"
    candidates = sorted(glob.glob(os.path.join(output_root, "4DGS_*")))
    if candidates:
        LOCAL_OUTPUT = candidates[-1]
    else:
        print("❌ HATA: Model klasörü bulunamadı!")
        sys.exit(1)

model_path = LOCAL_OUTPUT
NET_WIDTH = 128 # Checkpoint boyutu

cmd_args = [
    sys.executable, "export_perframe_3DGS.py",
    "--source_path", source_path,
    "--model_path", model_path,
    "--iteration", str(ITERATIONS),
    "--skip_video",
    "--net_width", str(NET_WIDTH)
]

print(f"📂 Model: {model_path}")
print("⏳ İşleniyor (Bu sefer kareleri tek tek sayacak)...")

work_dir = "/content/4DGaussians-Enhanced"
try:
    process = subprocess.Popen(
        cmd_args,
        cwd=work_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    # Logları izle
    for line in process.stdout:
        print(line, end="")

    process.wait()

except Exception as e:
    print(f"❌ Hata: {e}")

# --- 3. SONUÇLARI TOPLA VE DRIVE'A AT ---
# Scriptin içinde gördük: output_path = os.path.join(args.model_path,"gaussian_pertimestamp")
output_folder_name = "gaussian_pertimestamp"
ply_source_dir = os.path.join(model_path, output_folder_name)

if os.path.exists(ply_source_dir):
    ply_files = glob.glob(os.path.join(ply_source_dir, "*.ply"))
    count = len(ply_files)

    if count > 0:
        print("\n" + "="*60)
        print(f"✅ BAŞARILI! {count} adet PLY dosyası üretildi.")
        print(f"📁 Kaynak: {ply_source_dir}")

        # Drive'a Yedekle
        model_name = os.path.basename(model_path)
        drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")

        print(f"\n📤 Drive'a Yedekleniyor: {drive_ply_dir}")
        if os.path.exists(drive_ply_dir):
            shutil.rmtree(drive_ply_dir)
        shutil.copytree(ply_source_dir, drive_ply_dir)

        print("🎉 İŞLEM TAMAMLANDI!")
        print("="*60)
    else:
        print("\n⚠️ Klasör var ama içi boş. Garip.")
else:
    print(f"\n❌ HATA: Beklenen '{output_folder_name}' klasörü oluşmadı.")

🚑 SCRIPT TAMİRİ VE EXPORT BAŞLATMA
🔧 Hedef Dosya: /content/4DGaussians-Enhanced/export_perframe_3DGS.py
   🐛 Hatalı döngü bulundu, düzeltiliyor...
   ✅ Yama başarıyla uygulandı!

🎬 EXPORT START
📂 Model: /content/output/4DGS_20251222_1530
⏳ İşleniyor (Bu sefer kareleri tek tek sayacak)...
  File "/content/4DGaussians-Enhanced/export_perframe_3DGS.py", line 102
    cameras = scene.getTestCameras()
IndentationError: unexpected indent

✅ BAŞARILI! 1 adet PLY dosyası üretildi.
📁 Kaynak: /content/output/4DGS_20251222_1530/gaussian_pertimestamp

📤 Drive'a Yedekleniyor: /content/drive/MyDrive/4DGS_project/output/4DGS_20251222_1530/ply_sequence
🎉 İŞLEM TAMAMLANDI!


In [ ]:
import sys
import subprocess
import os
import shutil

print("🚀 Manuel düzeltme sonrası Export (Final Deneme)...")

DATASET_PATH = "/content/data/my_scene/colmap_output"
output_root = "/content/output"
candidates = sorted([os.path.join(output_root, d) for d in os.listdir(output_root) if "4DGS_" in d])

if candidates:
    MODEL_PATH = candidates[-1]

    cmd_args = [
        sys.executable, "export_perframe_3DGS.py",
        "--source_path", DATASET_PATH,
        "--model_path", MODEL_PATH,
        "--iteration", "80000",
        "--skip_video",
        "--net_width", "128"
    ]

    print(f"📂 Model: {MODEL_PATH}")
    print("⏳ İşleniyor...")

    process = subprocess.Popen(
        cmd_args,
        cwd="/content/4DGaussians-Enhanced",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    # SONUÇ
    output_folder = os.path.join(MODEL_PATH, "gaussian_pertimestamp")
    if os.path.exists(output_folder):
        count = len(os.listdir(output_folder))
        print(f"\n✅ İŞLEM SONUCU: {count} dosya üretildi.")

        if count > 1:
            # Drive'a at
            model_name = os.path.basename(MODEL_PATH)
            drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")
            if os.path.exists(drive_ply_dir): shutil.rmtree(drive_ply_dir)
            shutil.copytree(output_folder, drive_ply_dir)
            print(f"📤 Drive'a yedeklendi: {drive_ply_dir}")
    else:
        print("\n❌ Klasör oluşmadı.")
else:
    print("❌ Model bulunamadı.")

🚀 Manuel düzeltme sonrası Export (Final Deneme)...
📂 Model: /content/output/4DGS_20251222_1530
⏳ İşleniyor...
Looking for config file in /content/output/4DGS_20251222_1530/cfg_args
Config file found: /content/output/4DGS_20251222_1530/cfg_args
Rendering  /content/output/4DGS_20251222_1530
feature_dim: 128 [22/12 18:55:57]
Loading trained model at iteration 80000 [22/12 18:55:57]

Reading camera 1/8
Reading camera 2/8
Reading camera 3/8
Reading camera 4/8
Reading camera 5/8
Reading camera 6/8
Reading camera 7/8
Reading camera 8/8 [22/12 18:55:58]
Loading Training Cameras [22/12 18:55:58]
Loading Test Cameras [22/12 18:55:58]
Loading Video Cameras [22/12 18:55:58]
Deformation Net Set aabb [10.146437   2.8368807  7.1748652] [ -8.10082    -2.0269814 -11.853425 ] [22/12 18:55:58]
Voxel Plane: set aabb= Parameter containing:
tensor([[ 10.1464,   2.8369,   7.1749],
        [ -8.1008,  -2.0270, -11.8534]]) [22/12 18:55:58]
loading model from exists/content/output/4DGS_20251222_1530/point_cloud

In [ ]:
import os
import sys
import shutil
import subprocess

print("="*60)
print("🎬 CELL 13.6: 66 KARE EXPORTER (CONFIG FINDER)")
print("="*60)

# --- AYARLAR ---
TOTAL_FRAMES = 66
NET_WIDTH = 128
OUTPUT_ROOT = "/content/output"

# Model yolunu bul
candidates = sorted([os.path.join(OUTPUT_ROOT, d) for d in os.listdir(OUTPUT_ROOT) if "4DGS_" in d])
if not candidates:
    print("❌ Model bulunamadı!")
    sys.exit(1)
MODEL_PATH = candidates[-1]

# --- SCRİPT ---
script_content = f"""
import torch
import os
import sys
import numpy as np
from scene import Scene
from gaussian_renderer import GaussianModel
from utils.render_utils import get_state_at_time
from plyfile import PlyData, PlyElement

# --- YARDIMCI FONKSİYONLAR ---
def construct_list_of_attributes(feature_dc_shape, feature_rest_shape, scaling_shape, rotation_shape):
    l = ['x', 'y', 'z', 'nx', 'ny', 'nz']
    for i in range(feature_dc_shape[1]*feature_dc_shape[2]): l.append('f_dc_{{}}'.format(i))
    for i in range(feature_rest_shape[1]*feature_rest_shape[2]): l.append('f_rest_{{}}'.format(i))
    l.append('opacity')
    for i in range(scaling_shape[1]): l.append('scale_{{}}'.format(i))
    for i in range(rotation_shape[1]): l.append('rot_{{}}'.format(i))
    return l

def init_3DGaussians_ply(points, scales, rotations, opactiy, shs, feature_shape):
    xyz = points.detach().cpu().numpy()
    normals = np.zeros_like(xyz)
    feature_dc = shs[:,0:feature_shape[0],:]
    feature_rest = shs[:,feature_shape[0]:,:]
    f_dc = shs[:,:feature_shape[0],:].detach().transpose(1,2).flatten(start_dim=1).contiguous().cpu().numpy()
    f_rest = shs[:,feature_shape[0]:,:].detach().transpose(1,2).flatten(start_dim=1).contiguous().cpu().numpy()
    opacities = opactiy.detach().cpu().numpy()
    scale = scales.detach().cpu().numpy()
    rotation = rotations.detach().cpu().numpy()
    dtype_full = [(attribute, 'f4') for attribute in construct_list_of_attributes(feature_dc.shape, feature_rest.shape, scales.shape, rotations.shape)]
    elements = np.empty(xyz.shape[0], dtype=dtype_full)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
    elements[:] = list(map(tuple, attributes))
    el = PlyElement.describe(elements, 'vertex')
    return PlyData([el])

def find_config_file():
    # Repoyu tara ve kplanes_config.py dosyasını bul
    repo_path = "/content/4DGaussians-Enhanced"
    for root, dirs, files in os.walk(repo_path):
        if "kplanes_config.py" in files:
            return os.path.join(root, "kplanes_config.py")
    return None

def export_66_frames():
    model_path = "{MODEL_PATH}"
    target_frames = {TOTAL_FRAMES}

    # 1. ARGS SİMÜLASYONU
    class MockArgs:
        def __init__(self):
            self.source_path = "/content/data/my_scene/colmap_output"
            self.model_path = model_path
            self.images = "images"
            self.eval = False; self.llffhold = 8; self.resolution = -1
            self.white_background = False; self.data_device = "cuda"
            self.sh_degree = 3;
            self.net_width = {NET_WIDTH}
            self.timebase_pe = 4; self.defor_depth = 1; self.posebase_pe = 10
            self.scale_rotation_pe = 2; self.opacity_pe = 2; self.timenet_width = 64
            self.timenet_output = 32; self.bounds = 1.6; self.plane_tv_weight = 0.0001
            self.time_smoothness_weight = 0.01; self.l1_time_planes = 0.0001
            # Config dosyasını dinamik yükleyeceğiz
            self.kplanes_config = None
            self.multires = [1, 2, 4, 8]
            self.no_dx = False; self.no_grid = False; self.no_ds = False
            self.no_dr = False; self.no_do = False; self.no_dshs = False
            self.empty_voxel = False; self.grid_pe = 0
            self.static_mlp = False; self.apply_rotation = False

    args = MockArgs()

    # 2. CONFIG DOSYASINI BUL VE YÜKLE
    real_config_path = find_config_file()
    if real_config_path:
        print(f"📄 Config bulundu: {{real_config_path}}")
        config_dict = {{}}
        with open(real_config_path, 'r') as f:
            exec(f.read(), {{}}, config_dict)
        if 'grid_config' in config_dict:
            args.kplanes_config = config_dict['grid_config']
            print("✅ Grid ayarları yüklendi.")
        else:
            raise ValueError("Config dosyasında 'grid_config' değişkeni yok!")
    else:
        raise FileNotFoundError("kplanes_config.py dosyası repo içinde bulunamadı!")

    # 3. MODELİ BAŞLAT
    print("🚀 Model yükleniyor...")
    gaussians = GaussianModel(args.sh_degree, args)
    scene = Scene(args, gaussians, load_iteration=80000, shuffle=False)

    # 4. KAMERA VE DÖNGÜ
    cameras = scene.getTrainCameras()
    if not cameras: cameras = scene.getTestCameras()
    if not cameras:
        print("❌ Hata: Kamera yok."); return

    ref_cam = cameras[0]

    output_dir = os.path.join(model_path, "ply_sequence_66")
    os.makedirs(output_dir, exist_ok=True)

    print(f"🎬 {{target_frames}} Kare işleniyor...")

    for i in range(target_frames):
        normalized_time = i / (target_frames - 1)
        ref_cam.timestamp = normalized_time

        points, scales, rots, opacity, shs = get_state_at_time(gaussians, ref_cam)

        feature_dc_shape = gaussians._features_dc.shape[1]
        feature_rest_shape = gaussians._features_rest.shape[1]
        gs_ply = init_3DGaussians_ply(points, scales, rots, opacity, shs, [feature_dc_shape, feature_rest_shape])

        filename = f"frame_{{i:05d}}.ply"
        gs_ply.write(os.path.join(output_dir, filename))

        if i % 10 == 0:
            print(f"   Frame {{i}}/{{target_frames}}")

    print(f"🎉 BİTTİ: {{output_dir}}")

if __name__ == "__main__":
    with torch.no_grad():
        export_66_frames()
"""

# Scripti yaz
script_path = "/content/4DGaussians-Enhanced/export_manual_66.py"
with open(script_path, "w") as f:
    f.write(script_content)

print("📝 Script hazırlandı. Çalıştırılıyor...")

# Çalıştır
cmd = [sys.executable, script_path]
process = subprocess.Popen(cmd, cwd="/content/4DGaussians-Enhanced", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")
process.wait()

# Yedekle
output_local = os.path.join(MODEL_PATH, "ply_sequence_66")
if os.path.exists(output_local):
    count = len(os.listdir(output_local))
    print("\n" + "="*60)
    print(f"📊 FİNAL RAPOR: {count} adet PLY dosyası.")
    if count == TOTAL_FRAMES:
        model_name = os.path.basename(MODEL_PATH)
        drive_dest = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence_66")
        print(f"📤 Drive'a aktarılıyor...")
        if os.path.exists(drive_dest): shutil.rmtree(drive_dest)
        shutil.copytree(output_local, drive_dest)
        print("✅ YEDEKLEME TAMAMLANDI.")
    else:
        print(f"⚠️ Eksik dosya: {count}/{TOTAL_FRAMES}")
else:
    print("❌ HATA: Klasör oluşmadı.")

🎬 CELL 13.6: 66 KARE EXPORTER (CONFIG FINDER)
📝 Script hazırlandı. Çalıştırılıyor...
Traceback (most recent call last):
  File "/content/4DGaussians-Enhanced/export_manual_66.py", line 127, in <module>
    export_66_frames()
  File "/content/4DGaussians-Enhanced/export_manual_66.py", line 87, in export_66_frames
    raise FileNotFoundError("kplanes_config.py dosyası repo içinde bulunamadı!")
FileNotFoundError: kplanes_config.py dosyası repo içinde bulunamadı!
❌ HATA: Klasör oluşmadı.
